# 01F — Lead Features Hybrid Optimizer
## bd_replica_crm · recuperar semántica exacta sin perder velocidad

Resultado previo:
- 01D LATERAL: equivalencia 100%, speedup 0.81x
- 01E DAILY: equivalencia 0%, speedup 3.17x

Objetivo:
- equivalencia point-in-time >= 99.9%
- speedup >= 1.5x

Estrategia híbrida:
- cliente: timestamp exacto
- proyecto/asesor/global: días completos preagregados
- día/frontera: corrección exacta intradía
- maduración 14d/60d: respetar timestamp exacto

Read-only por defecto.


In [ ]:
from __future__ import annotations
import sys, time
from pathlib import Path
import pandas as pd
import numpy as np

cwd=Path.cwd().resolve()
PROJECT_ROOT=cwd if (cwd/"pyproject.toml").exists() else cwd.parent
if not (PROJECT_ROOT/"pyproject.toml").exists():
    raise RuntimeError("Ejecuta este notebook dentro de bd_replica_crm.")

SRC=PROJECT_ROOT/"src"
if str(SRC) not in sys.path:
    sys.path.insert(0,str(SRC))

from replica_cygnus.settings import load_settings
from replica_cygnus.connections import connect_postgres
from replica_cygnus.lead_scoring.config import load_lead_scoring_config

settings=load_settings(PROJECT_ROOT)
config_path=PROJECT_ROOT/"config"/"lead_scoring.yml"
if not config_path.exists():
    config_path=PROJECT_ROOT/"config"/"lead_scoring.example.yml"

cfg=load_lead_scoring_config(config_path)
conn=connect_postgres(settings)

pd.set_option("display.max_columns",180)
pd.set_option("display.max_rows",180)
pd.set_option("display.width",260)

def df(sql,params=None):
    return pd.read_sql_query(sql,conn,params=params)

SEP_H=int(cfg.sep_horizon_days)
MINUTA_H=int(cfg.minuta_horizon_days)
SCORE_WINDOW=int(cfg.score_window_days)

print("DB:",settings.postgres.database)
print("score_window_days:",SCORE_WINDOW)


DB: medallio_dw
score_window_days: 14


## 1. Estado actual


In [ ]:
status=df(f"""
SELECT
 COUNT(*) AS total,
 COUNT(*) FILTER (WHERE features_refreshed_at IS NULL) AS pending,
 COUNT(*) FILTER (
   WHERE features_refreshed_at IS NULL
     AND decision_at>=current_date-({SCORE_WINDOW}*interval '1 day')
 ) AS pending_live
FROM features.lead_evidence
""")
status.T


C:\Users\dinat\AppData\Local\Temp\ipykernel_8744\921070025.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


,0
total,206043
pending,206043
pending_live,2026


## 2. Índices actuales


In [ ]:
indexes=df("""
SELECT indexname,indexdef
FROM pg_indexes
WHERE schemaname='features' AND tablename='lead_evidence'
ORDER BY indexname
""")
indexes


C:\Users\dinat\AppData\Local\Temp\ipykernel_8744\921070025.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


,indexname,indexdef
0,ix_lead_evidence_advisor_time,CREATE INDEX ix_lead_evidence_advisor_time ON ...
1,ix_lead_evidence_decision_at,CREATE INDEX ix_lead_evidence_decision_at ON f...
2,ix_lead_evidence_document_time,CREATE INDEX ix_lead_evidence_document_time ON...
3,ix_lead_evidence_project_time,CREATE INDEX ix_lead_evidence_project_time ON ...
4,lead_evidence_pkey,CREATE UNIQUE INDEX lead_evidence_pkey ON feat...


## 3. Baseline exacto


In [ ]:
def baseline_query(limit_n=100):
    return f"""
WITH target AS (
 SELECT *
 FROM features.lead_evidence
 WHERE features_refreshed_at IS NULL
   AND decision_at>=current_date-({SCORE_WINDOW}*interval '1 day')
 ORDER BY decision_at,evidence_key
 LIMIT {int(limit_n)}
)
SELECT
 e.evidence_key,e.decision_at,e.documento_cliente,e.codigo_proyecto,e.asesor,
 COALESCE((SELECT COUNT(*) FROM features.lead_evidence p
   WHERE p.documento_cliente=e.documento_cliente
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '90 days'),0) AS client_prior_assignments_90d,
 (SELECT EXTRACT(EPOCH FROM (e.decision_at-MAX(p.decision_at)))/86400.0
   FROM features.lead_evidence p
   WHERE p.documento_cliente=e.documento_cliente
     AND p.decision_at<e.decision_at) AS days_since_previous_assignment,
 COALESCE((SELECT COUNT(*) FROM features.lead_evidence p
   WHERE p.codigo_proyecto=e.codigo_proyecto
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '90 days'),0) AS project_leads_90d,
 (SELECT AVG(p.separacion_14d::double precision)
   FROM features.lead_evidence p
   WHERE p.codigo_proyecto=e.codigo_proyecto
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '90 days'
     AND p.separacion_14d IS NOT NULL
     AND p.decision_at+interval '{SEP_H} days'<=e.decision_at) AS project_sep_rate_90d,
 (SELECT AVG(p.minuta_60d::double precision)
   FROM features.lead_evidence p
   WHERE p.codigo_proyecto=e.codigo_proyecto
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '180 days'
     AND p.minuta_60d IS NOT NULL
     AND p.decision_at+interval '{MINUTA_H} days'<=e.decision_at) AS project_minuta_rate_180d,
 CASE WHEN e.asesor IS NULL THEN NULL ELSE COALESCE((SELECT COUNT(*)
   FROM features.lead_evidence p
   WHERE p.asesor=e.asesor
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '90 days'),0) END AS advisor_leads_90d,
 CASE WHEN e.asesor IS NULL THEN NULL ELSE (SELECT AVG(p.separacion_14d::double precision)
   FROM features.lead_evidence p
   WHERE p.asesor=e.asesor
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '90 days'
     AND p.separacion_14d IS NOT NULL
     AND p.decision_at+interval '{SEP_H} days'<=e.decision_at) END AS advisor_sep_rate_90d,
 CASE WHEN e.asesor IS NULL THEN NULL ELSE (SELECT AVG(p.minuta_60d::double precision)
   FROM features.lead_evidence p
   WHERE p.asesor=e.asesor
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '180 days'
     AND p.minuta_60d IS NOT NULL
     AND p.decision_at+interval '{MINUTA_H} days'<=e.decision_at) END AS advisor_minuta_rate_180d,
 (SELECT AVG(p.separacion_14d::double precision)
   FROM features.lead_evidence p
   WHERE p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '90 days'
     AND p.separacion_14d IS NOT NULL
     AND p.decision_at+interval '{SEP_H} days'<=e.decision_at) AS global_sep_rate_90d,
 (SELECT AVG(p.minuta_60d::double precision)
   FROM features.lead_evidence p
   WHERE p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '180 days'
     AND p.minuta_60d IS NOT NULL
     AND p.decision_at+interval '{MINUTA_H} days'<=e.decision_at) AS global_minuta_rate_180d
FROM target e
ORDER BY e.decision_at,e.evidence_key
"""


## 4. Híbrido: días completos + frontera exacta


In [ ]:
def hybrid_query(limit_n=100):
    return f"""
WITH target AS (
 SELECT *
 FROM features.lead_evidence
 WHERE features_refreshed_at IS NULL
   AND decision_at>=current_date-({SCORE_WINDOW}*interval '1 day')
 ORDER BY decision_at,evidence_key
 LIMIT {int(limit_n)}
),
project_daily AS (
 SELECT codigo_proyecto,decision_at::date d,
        COUNT(*) leads_day,
        SUM(separacion_14d) FILTER (WHERE separacion_14d IS NOT NULL) sep_pos_day,
        COUNT(separacion_14d) sep_matured_day,
        SUM(minuta_60d) FILTER (WHERE minuta_60d IS NOT NULL) minuta_pos_day,
        COUNT(minuta_60d) minuta_matured_day
 FROM features.lead_evidence
 GROUP BY codigo_proyecto,decision_at::date
),
advisor_daily AS (
 SELECT asesor,decision_at::date d,
        COUNT(*) leads_day,
        SUM(separacion_14d) FILTER (WHERE separacion_14d IS NOT NULL) sep_pos_day,
        COUNT(separacion_14d) sep_matured_day,
        SUM(minuta_60d) FILTER (WHERE minuta_60d IS NOT NULL) minuta_pos_day,
        COUNT(minuta_60d) minuta_matured_day
 FROM features.lead_evidence
 WHERE asesor IS NOT NULL
 GROUP BY asesor,decision_at::date
),
global_daily AS (
 SELECT decision_at::date d,
        SUM(separacion_14d) FILTER (WHERE separacion_14d IS NOT NULL) sep_pos_day,
        COUNT(separacion_14d) sep_matured_day,
        SUM(minuta_60d) FILTER (WHERE minuta_60d IS NOT NULL) minuta_pos_day,
        COUNT(minuta_60d) minuta_matured_day
 FROM features.lead_evidence
 GROUP BY decision_at::date
)
SELECT
 e.evidence_key,e.decision_at,e.documento_cliente,e.codigo_proyecto,e.asesor,

 -- cliente exacto
 COALESCE((SELECT COUNT(*) FROM features.lead_evidence p
   WHERE p.documento_cliente=e.documento_cliente
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '90 days'),0)
 AS client_prior_assignments_90d,

 (SELECT EXTRACT(EPOCH FROM (e.decision_at-MAX(p.decision_at)))/86400.0
   FROM features.lead_evidence p
   WHERE p.documento_cliente=e.documento_cliente
     AND p.decision_at<e.decision_at)
 AS days_since_previous_assignment,

 -- proyecto leads 90d: días completos + intradía
 COALESCE((
   SELECT SUM(pd.leads_day)
   FROM project_daily pd
   WHERE pd.codigo_proyecto=e.codigo_proyecto
     AND pd.d>=e.decision_at::date-90
     AND pd.d<e.decision_at::date
 ),0)
 + COALESCE((
   SELECT COUNT(*)
   FROM features.lead_evidence p
   WHERE p.codigo_proyecto=e.codigo_proyecto
     AND p.decision_at>=date_trunc('day',e.decision_at)
     AND p.decision_at<e.decision_at
 ),0)
 AS project_leads_90d,

 -- proyecto sep exacto en fronteras
 (
   COALESCE((
     SELECT SUM(pd.sep_pos_day)::double precision
     FROM project_daily pd
     WHERE pd.codigo_proyecto=e.codigo_proyecto
       AND pd.d>=e.decision_at::date-90
       AND pd.d<(e.decision_at-interval '{SEP_H} days')::date
   ),0)
   +
   COALESCE((
     SELECT SUM(p.separacion_14d)::double precision
     FROM features.lead_evidence p
     WHERE p.codigo_proyecto=e.codigo_proyecto
       AND p.decision_at>=date_trunc('day',e.decision_at-interval '{SEP_H} days')
       AND p.decision_at<=e.decision_at-interval '{SEP_H} days'
       AND p.decision_at>=e.decision_at-interval '90 days'
       AND p.separacion_14d IS NOT NULL
   ),0)
 )
 / NULLIF(
   COALESCE((
     SELECT SUM(pd.sep_matured_day)
     FROM project_daily pd
     WHERE pd.codigo_proyecto=e.codigo_proyecto
       AND pd.d>=e.decision_at::date-90
       AND pd.d<(e.decision_at-interval '{SEP_H} days')::date
   ),0)
   +
   COALESCE((
     SELECT COUNT(p.separacion_14d)
     FROM features.lead_evidence p
     WHERE p.codigo_proyecto=e.codigo_proyecto
       AND p.decision_at>=date_trunc('day',e.decision_at-interval '{SEP_H} days')
       AND p.decision_at<=e.decision_at-interval '{SEP_H} days'
       AND p.decision_at>=e.decision_at-interval '90 days'
       AND p.separacion_14d IS NOT NULL
   ),0),0)
 AS project_sep_rate_90d,

 -- proyecto minuta exacta en frontera
 (
   COALESCE((
     SELECT SUM(pd.minuta_pos_day)::double precision
     FROM project_daily pd
     WHERE pd.codigo_proyecto=e.codigo_proyecto
       AND pd.d>=e.decision_at::date-180
       AND pd.d<(e.decision_at-interval '{MINUTA_H} days')::date
   ),0)
   +
   COALESCE((
     SELECT SUM(p.minuta_60d)::double precision
     FROM features.lead_evidence p
     WHERE p.codigo_proyecto=e.codigo_proyecto
       AND p.decision_at>=date_trunc('day',e.decision_at-interval '{MINUTA_H} days')
       AND p.decision_at<=e.decision_at-interval '{MINUTA_H} days'
       AND p.decision_at>=e.decision_at-interval '180 days'
       AND p.minuta_60d IS NOT NULL
   ),0)
 )
 / NULLIF(
   COALESCE((
     SELECT SUM(pd.minuta_matured_day)
     FROM project_daily pd
     WHERE pd.codigo_proyecto=e.codigo_proyecto
       AND pd.d>=e.decision_at::date-180
       AND pd.d<(e.decision_at-interval '{MINUTA_H} days')::date
   ),0)
   +
   COALESCE((
     SELECT COUNT(p.minuta_60d)
     FROM features.lead_evidence p
     WHERE p.codigo_proyecto=e.codigo_proyecto
       AND p.decision_at>=date_trunc('day',e.decision_at-interval '{MINUTA_H} days')
       AND p.decision_at<=e.decision_at-interval '{MINUTA_H} days'
       AND p.decision_at>=e.decision_at-interval '180 days'
       AND p.minuta_60d IS NOT NULL
   ),0),0)
 AS project_minuta_rate_180d,

 -- asesor leads
 CASE WHEN e.asesor IS NULL THEN NULL ELSE
 COALESCE((
   SELECT SUM(ad.leads_day)
   FROM advisor_daily ad
   WHERE ad.asesor=e.asesor
     AND ad.d>=e.decision_at::date-90
     AND ad.d<e.decision_at::date
 ),0)
 + COALESCE((
   SELECT COUNT(*)
   FROM features.lead_evidence p
   WHERE p.asesor=e.asesor
     AND p.decision_at>=date_trunc('day',e.decision_at)
     AND p.decision_at<e.decision_at
 ),0) END
 AS advisor_leads_90d,

 -- asesor sep
 CASE WHEN e.asesor IS NULL THEN NULL ELSE
 (
   COALESCE((
     SELECT SUM(ad.sep_pos_day)::double precision
     FROM advisor_daily ad
     WHERE ad.asesor=e.asesor
       AND ad.d>=e.decision_at::date-90
       AND ad.d<(e.decision_at-interval '{SEP_H} days')::date
   ),0)
   +
   COALESCE((
     SELECT SUM(p.separacion_14d)::double precision
     FROM features.lead_evidence p
     WHERE p.asesor=e.asesor
       AND p.decision_at>=date_trunc('day',e.decision_at-interval '{SEP_H} days')
       AND p.decision_at<=e.decision_at-interval '{SEP_H} days'
       AND p.decision_at>=e.decision_at-interval '90 days'
       AND p.separacion_14d IS NOT NULL
   ),0)
 )
 / NULLIF(
   COALESCE((
     SELECT SUM(ad.sep_matured_day)
     FROM advisor_daily ad
     WHERE ad.asesor=e.asesor
       AND ad.d>=e.decision_at::date-90
       AND ad.d<(e.decision_at-interval '{SEP_H} days')::date
   ),0)
   +
   COALESCE((
     SELECT COUNT(p.separacion_14d)
     FROM features.lead_evidence p
     WHERE p.asesor=e.asesor
       AND p.decision_at>=date_trunc('day',e.decision_at-interval '{SEP_H} days')
       AND p.decision_at<=e.decision_at-interval '{SEP_H} days'
       AND p.decision_at>=e.decision_at-interval '90 days'
       AND p.separacion_14d IS NOT NULL
   ),0),0)
 END AS advisor_sep_rate_90d,

 -- asesor minuta
 CASE WHEN e.asesor IS NULL THEN NULL ELSE
 (
   COALESCE((
     SELECT SUM(ad.minuta_pos_day)::double precision
     FROM advisor_daily ad
     WHERE ad.asesor=e.asesor
       AND ad.d>=e.decision_at::date-180
       AND ad.d<(e.decision_at-interval '{MINUTA_H} days')::date
   ),0)
   +
   COALESCE((
     SELECT SUM(p.minuta_60d)::double precision
     FROM features.lead_evidence p
     WHERE p.asesor=e.asesor
       AND p.decision_at>=date_trunc('day',e.decision_at-interval '{MINUTA_H} days')
       AND p.decision_at<=e.decision_at-interval '{MINUTA_H} days'
       AND p.decision_at>=e.decision_at-interval '180 days'
       AND p.minuta_60d IS NOT NULL
   ),0)
 )
 / NULLIF(
   COALESCE((
     SELECT SUM(ad.minuta_matured_day)
     FROM advisor_daily ad
     WHERE ad.asesor=e.asesor
       AND ad.d>=e.decision_at::date-180
       AND ad.d<(e.decision_at-interval '{MINUTA_H} days')::date
   ),0)
   +
   COALESCE((
     SELECT COUNT(p.minuta_60d)
     FROM features.lead_evidence p
     WHERE p.asesor=e.asesor
       AND p.decision_at>=date_trunc('day',e.decision_at-interval '{MINUTA_H} days')
       AND p.decision_at<=e.decision_at-interval '{MINUTA_H} days'
       AND p.decision_at>=e.decision_at-interval '180 days'
       AND p.minuta_60d IS NOT NULL
   ),0),0)
 END AS advisor_minuta_rate_180d,

 -- global sep
 (
   COALESCE((
     SELECT SUM(gd.sep_pos_day)::double precision
     FROM global_daily gd
     WHERE gd.d>=e.decision_at::date-90
       AND gd.d<(e.decision_at-interval '{SEP_H} days')::date
   ),0)
   +
   COALESCE((
     SELECT SUM(p.separacion_14d)::double precision
     FROM features.lead_evidence p
     WHERE p.decision_at>=date_trunc('day',e.decision_at-interval '{SEP_H} days')
       AND p.decision_at<=e.decision_at-interval '{SEP_H} days'
       AND p.decision_at>=e.decision_at-interval '90 days'
       AND p.separacion_14d IS NOT NULL
   ),0)
 )
 / NULLIF(
   COALESCE((
     SELECT SUM(gd.sep_matured_day)
     FROM global_daily gd
     WHERE gd.d>=e.decision_at::date-90
       AND gd.d<(e.decision_at-interval '{SEP_H} days')::date
   ),0)
   +
   COALESCE((
     SELECT COUNT(p.separacion_14d)
     FROM features.lead_evidence p
     WHERE p.decision_at>=date_trunc('day',e.decision_at-interval '{SEP_H} days')
       AND p.decision_at<=e.decision_at-interval '{SEP_H} days'
       AND p.decision_at>=e.decision_at-interval '90 days'
       AND p.separacion_14d IS NOT NULL
   ),0),0)
 AS global_sep_rate_90d,

 -- global minuta
 (
   COALESCE((
     SELECT SUM(gd.minuta_pos_day)::double precision
     FROM global_daily gd
     WHERE gd.d>=e.decision_at::date-180
       AND gd.d<(e.decision_at-interval '{MINUTA_H} days')::date
   ),0)
   +
   COALESCE((
     SELECT SUM(p.minuta_60d)::double precision
     FROM features.lead_evidence p
     WHERE p.decision_at>=date_trunc('day',e.decision_at-interval '{MINUTA_H} days')
       AND p.decision_at<=e.decision_at-interval '{MINUTA_H} days'
       AND p.decision_at>=e.decision_at-interval '180 days'
       AND p.minuta_60d IS NOT NULL
   ),0)
 )
 / NULLIF(
   COALESCE((
     SELECT SUM(gd.minuta_matured_day)
     FROM global_daily gd
     WHERE gd.d>=e.decision_at::date-180
       AND gd.d<(e.decision_at-interval '{MINUTA_H} days')::date
   ),0)
   +
   COALESCE((
     SELECT COUNT(p.minuta_60d)
     FROM features.lead_evidence p
     WHERE p.decision_at>=date_trunc('day',e.decision_at-interval '{MINUTA_H} days')
       AND p.decision_at<=e.decision_at-interval '{MINUTA_H} days'
       AND p.decision_at>=e.decision_at-interval '180 days'
       AND p.minuta_60d IS NOT NULL
   ),0),0)
 AS global_minuta_rate_180d

FROM target e
ORDER BY e.decision_at,e.evidence_key
"""


## 5. Benchmark progresivo


In [ ]:
BENCH_SIZES=[100,500,1000]
rows=[]
for n in BENCH_SIZES:
    for name,fn in [("current_correlated",baseline_query),("hybrid",hybrid_query)]:
        t0=time.perf_counter()
        try:
            out=df(fn(n)); sec=time.perf_counter()-t0
            rows.append({"n":n,"method":name,"seconds":sec,"rows":len(out),"rows_per_second":len(out)/sec if sec else np.nan})
            print(f"{name} n={n}: {sec:.3f}s")
        except Exception as exc:
            conn.rollback()
            rows.append({"n":n,"method":name,"seconds":np.nan,"rows":0,"error":repr(exc)})
benchmark=pd.DataFrame(rows)
benchmark


C:\Users\dinat\AppData\Local\Temp\ipykernel_8744\921070025.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


current_correlated n=100: 3.613s


C:\Users\dinat\AppData\Local\Temp\ipykernel_8744\921070025.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


hybrid n=100: 4.508s


C:\Users\dinat\AppData\Local\Temp\ipykernel_8744\921070025.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


...

C:\Users\dinat\AppData\Local\Temp\ipykernel_8744\921070025.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


current_correlated n=1000: 72.106s


C:\Users\dinat\AppData\Local\Temp\ipykernel_8744\921070025.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


hybrid n=1000: 44.117s


,n,method,seconds,rows,rows_per_second
0,100,current_correlated,3.613146,100,27.676713
1,100,hybrid,4.507512,100,22.185185
2,500,current_correlated,34.064021,500,14.678244
3,500,hybrid,19.422997,500,25.742681
4,1000,current_correlated,72.105542,1000,13.868559
5,1000,hybrid,44.117091,1000,22.666952


## 6. Speedup


In [ ]:
pivot=benchmark.pivot(index="n",columns="method",values="seconds").reset_index()
if {"current_correlated","hybrid"}.issubset(pivot.columns):
    pivot["speedup_x"]=pivot["current_correlated"]/pivot["hybrid"]
pivot


method,n,current_correlated,hybrid,speedup_x
0,100,3.613146,4.507512,0.801583
1,500,34.064021,19.422997,1.753798
2,1000,72.105542,44.117091,1.634413


## 7. Equivalencia feature por feature


In [ ]:
N_CHECK=100
base=df(baseline_query(N_CHECK))
hyb=df(hybrid_query(N_CHECK))

features=[
"client_prior_assignments_90d","days_since_previous_assignment",
"project_leads_90d","project_sep_rate_90d","project_minuta_rate_180d",
"advisor_leads_90d","advisor_sep_rate_90d","advisor_minuta_rate_180d",
"global_sep_rate_90d","global_minuta_rate_180d"
]
m=base.merge(hyb,on="evidence_key",suffixes=("_base","_hybrid"))
checks=[]
for c in features:
    a=pd.to_numeric(m[f"{c}_base"],errors="coerce")
    b=pd.to_numeric(m[f"{c}_hybrid"],errors="coerce")
    ok=np.isclose(a.fillna(-999999),b.fillna(-999999),rtol=1e-9,atol=1e-9)
    checks.append({"feature":c,"match_pct":ok.mean(),"rows":len(ok),"max_abs_diff":float((a-b).abs().max()) if len(a) else np.nan})
equivalence=pd.DataFrame(checks)
equivalence


C:\Users\dinat\AppData\Local\Temp\ipykernel_8744\921070025.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)
C:\Users\dinat\AppData\Local\Temp\ipykernel_8744\921070025.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


,feature,match_pct,rows,max_abs_diff
0,client_prior_assignments_90d,1.00,100,0.000000
1,days_since_previous_assignment,1.00,100,0.000000
2,project_leads_90d,0.06,100,28.000000
3,project_sep_rate_90d,0.07,100,0.000115
4,project_minuta_rate_180d,0.05,100,0.000106
5,advisor_leads_90d,1.00,100,NaN
6,advisor_sep_rate_90d,1.00,100,NaN
7,advisor_minuta_rate_180d,1.00,100,NaN
8,global_sep_rate_90d,0.00,100,0.000088
9,global_minuta_rate_180d,0.00,100,0.000052


## 8. Diferencias concretas


In [ ]:
diffs=[]
for c in features:
    a=pd.to_numeric(m[f"{c}_base"],errors="coerce")
    b=pd.to_numeric(m[f"{c}_hybrid"],errors="coerce")
    ok=np.isclose(a.fillna(-999999),b.fillna(-999999),rtol=1e-9,atol=1e-9)
    bad=m.loc[~ok,["evidence_key","decision_at",f"{c}_base",f"{c}_hybrid"]].head(20).copy()
    if len(bad):
        bad["feature"]=c
        diffs.append(bad)

if diffs:
    diff_examples=pd.concat(diffs,ignore_index=True)
    display(diff_examples.head(200))
else:
    diff_examples=pd.DataFrame()
    print("Sin diferencias.")


KeyError: "['decision_at'] not in index"

## 9. EXPLAIN comparativo


In [ ]:
for name,q in [("CURRENT",baseline_query(100)),("HYBRID",hybrid_query(100))]:
    print("\n###",name)
    p=df("EXPLAIN "+q)
    print("\n".join(p.iloc[:,0].astype(str)))


## 10. EXPLAIN ANALYZE opcional


In [ ]:
RUN_EXPLAIN_ANALYZE=False
if RUN_EXPLAIN_ANALYZE:
    for name,q in [("CURRENT",baseline_query(100)),("HYBRID",hybrid_query(100))]:
        print("\n###",name)
        p=df("EXPLAIN (ANALYZE,BUFFERS) "+q)
        print("\n".join(p.iloc[:,0].astype(str)))
else:
    print("RUN_EXPLAIN_ANALYZE=False")


## 11. Estimación runtime LIVE


In [ ]:
pending_live=int(status.iloc[0]["pending_live"])
est=[]
for method in benchmark["method"].unique():
    x=benchmark[(benchmark["method"]==method)&benchmark["rows_per_second"].notna()&benchmark["rows_per_second"].gt(0)]
    if len(x):
        r=x.sort_values("n").iloc[-1]
        rps=float(r["rows_per_second"])
        sec=pending_live/rps
        est.append({"method":method,"reference_rps":rps,"estimated_seconds_live":sec,"estimated_minutes_live":sec/60})
live_estimates=pd.DataFrame(est)
live_estimates


## 12. Full LIVE opcional


In [ ]:
RUN_FULL_LIVE=False
if RUN_FULL_LIVE:
    n=int(status.iloc[0]["pending_live"])
    full=[]
    for name,fn in [("current_correlated",baseline_query),("hybrid",hybrid_query)]:
        t0=time.perf_counter()
        out=df(fn(n)); sec=time.perf_counter()-t0
        full.append({"method":name,"rows":len(out),"seconds":sec,"minutes":sec/60})
    full_live=pd.DataFrame(full)
    display(full_live)
else:
    print("RUN_FULL_LIVE=False")


## 13. Índices candidatos


In [ ]:
index_candidates=pd.DataFrame([
{"index":"(documento_cliente, decision_at)","purpose":"cliente exacto"},
{"index":"(codigo_proyecto, decision_at)","purpose":"fronteras proyecto"},
{"index":"(asesor, decision_at)","purpose":"fronteras asesor"},
{"index":"(decision_at)","purpose":"fronteras globales / live"}
])
index_candidates


## 14. Gates


In [ ]:
gates=[]
def add(name,passed,detail):
    gates.append({"gate":name,"status":"PASS" if passed else "FAIL","detail":detail})

mm=float(equivalence["match_pct"].min()) if len(equivalence) else np.nan
add("Equivalencia point-in-time",pd.notna(mm) and mm>=0.999,f"min_match={mm:.3%}" if pd.notna(mm) else "N/A")

pv=benchmark.pivot(index="n",columns="method",values="seconds").dropna()
if len(pv) and {"current_correlated","hybrid"}.issubset(pv.columns):
    r=pv.iloc[-1]
    sx=float(r["current_correlated"]/r["hybrid"])
    add("Speedup material",sx>=1.5,f"speedup={sx:.2f}x")

add("LIVE incremental",int(status.iloc[0]["pending_live"])>0,f"pending_live={int(status.iloc[0]['pending_live']):,}")

gate_table=pd.DataFrame(gates)
gate_table


## 15. Smart insights


In [ ]:
print("=== 01F LEAD FEATURES HYBRID OPTIMIZER ===")
print(f"1. Pendientes históricos: {int(status.iloc[0]['pending']):,}")
print(f"2. Pendientes live: {int(status.iloc[0]['pending_live']):,}")
if len(equivalence):
    print(f"3. Equivalencia mínima: {equivalence['match_pct'].min():.1%}")
if len(pv) and {"current_correlated","hybrid"}.issubset(pv.columns):
    r=pv.iloc[-1]
    print(f"4. Speedup mayor benchmark: {float(r['current_correlated']/r['hybrid']):.2f}x")
display(live_estimates)


## 16. Regla de promoción

Solo considerar implementación productiva si:

```text
Equivalencia point-in-time >= 99.9%
Speedup material          >= 1.5x
```


## 17. Cambios deshabilitados


In [ ]:
CREATE_PERSISTENT_DAILY_TABLES=False
APPLY_INDEXES=False
APPLY_HYBRID_FEATURES=False

print("CREATE_PERSISTENT_DAILY_TABLES =",CREATE_PERSISTENT_DAILY_TABLES)
print("APPLY_INDEXES =",APPLY_INDEXES)
print("APPLY_HYBRID_FEATURES =",APPLY_HYBRID_FEATURES)


In [ ]:
conn.close(); print("Conexión cerrada.")
